# PopQA bằng code gốc: Qwen2.5-3B (bản base), vanilla và BM25

Phần mở rộng của đồ án CS221 (issue #30), tiếp theo lượt Qwen2.5 0.5B/1.5B. Hai lượt, mỗi lượt đủ 14.267 câu PopQA, cùng code gốc `run_model.py` (fp32), seed 633, 15 ví dụ mẫu, cùng thứ tự câu với các lượt GPT-Neo/OPT/Qwen của nhóm:

| GPU | Lượt chạy | Ước tính |
|---|---|---|
| 0 | Qwen2.5-3B · BM25 (15 ví dụ mẫu kèm đoạn văn) | khoảng 17–19 giờ, cần 2 phiên |
| 1 | Qwen2.5-3B · vanilla | khoảng 6 giờ |

Cửa sổ ngữ cảnh giữ **2048 token như GPT-Neo/OPT trong paper** (cắt từ trái, dành 15 token cho câu trả lời). Có cột `conf` (độ tự tin của câu trả lời). Qwen2.5-3B ở fp32 chiếm khoảng 12,4 GB trên GPU T4 15 GB; nếu thiếu bộ nhớ, lỗi hiện ngay ở lượt thử 300 câu (khoảng 20 phút đầu).

## Cách chạy (người chạy chỉ cần làm 4 bước)
1. Kaggle → **Create → New Notebook → File → Import Notebook**, chọn file này.
2. Khung bên phải → **Session options**: *Accelerator* = **GPU T4 ×2**; *Internet* = **On** (tài khoản cần xác minh số điện thoại).
3. **Add Input** → tìm dataset `vanductan/popqa-bm25-retrieval` → Add. (Code gốc và PopQA notebook tự tải từ GitHub của tác giả.)
4. **Save Version → Save & Run All (Commit)**. Có thể tắt máy; Kaggle tự chạy.

**Phiên 1** tự dừng gọn sau 11,4 giờ, lúc đó lượt BM25 mới xong khoảng 2/3.

**Phiên 2** (chạy tiếp BM25):
1. Vào tab **Output** của phiên 1, bấm **New Dataset** từ file `popqa-qwen-checkpoint.zip`, rồi **Add Input** dataset đó vào notebook.
2. Ở cell có dòng `TIME_BUDGET_HOURS = 11.4`, sửa thành **số giờ GPU còn lại trong tuần trừ 0,7** (ví dụ còn 7 giờ 50 phút thì ghi `7.1`). Như vậy phiên tự dừng và lưu checkpoint trước khi hết quota; phần còn lại chạy tuần sau.
3. **Save Version → Save & Run All**. Notebook tự chạy tiếp từ câu đã lưu.

Chạy xong: để notebook ở chế độ **Public** (Share → Public), hoặc tải *Output* về gửi lại người nhờ chạy.


In [1]:
import sys, subprocess, time
# Giờ bắt đầu phiên: lượt full dừng gọn trước giới hạn 12 giờ của Kaggle.
SESSION_START = time.time()
# Ghim đúng bản đã chạy pilot: đổi bản thư viện thì RUN_KEY đổi và checkpoint phiên trước bị từ chối.
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'openai', 'jsonlines==4.0.0', 'numpy', 'pandas',
    'matplotlib', 'seaborn', 'tqdm',
])

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


0

## 1. Chuẩn bị GPU, mã nguồn và dữ liệu

In [2]:
from pathlib import Path
import shutil, zipfile, os, hashlib
import torch
import pandas as pd
from IPython.display import display

assert torch.cuda.is_available(), 'Enable GPU in Kaggle settings.'
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), 'GiB')

work = Path('/kaggle/working/popqa-qwen')
required = ['run_model.py', 'util_clm.py', 'data/popQA.tsv']
input_root = Path('/kaggle/input')
archives = list(input_root.rglob('adaptive-retrieval-kaggle.zip'))
if archives:
    assert len(archives) == 1, 'Multiple source ZIPs: select the intended one explicitly.'
    with zipfile.ZipFile(archives[0]) as archive:
        for rel in required:
            members = [n for n in archive.namelist() if n.endswith('/' + rel) or n == rel]
            assert len(members) == 1, f'Expected exactly one {rel}'
            target = work / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(members[0]))
else:
    candidates = [p.parent for p in input_root.rglob('run_model.py')
                  if all((p.parent / rel).is_file() for rel in required)]
    if not candidates:
        previous = Path('/kaggle/working/adaptive-retrieval')
        if all((previous / rel).is_file() for rel in required):
            candidates = [previous]
    if candidates:
        assert len(candidates) == 1, 'Có nhiều bộ mã nguồn trong Input; chỉ giữ một.'
        for rel in required:
            target = work / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidates[0] / rel, target)
    else:
        # Không có dataset mã nguồn trong Input: tải đúng bản của tác giả (commit 8172ab0) từ GitHub.
        # Cần bật Internet. Mã hash của run_model.py và popQA.tsv được kiểm ngay sau đây.
        import urllib.request
        AUTHOR_RAW = 'https://raw.githubusercontent.com/AlexTMallen/adaptive-retrieval/8172ab04794a1651488207c230286b0b3786d736/'
        for rel in required:
            target = work / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(AUTHOR_RAW + rel, target)
        print('Đã tải mã nguồn và PopQA của tác giả từ GitHub (commit 8172ab0).')

os.chdir(work)
Path('results').mkdir(exist_ok=True)
Path('logs').mkdir(exist_ok=True)
questions = pd.read_csv('data/popQA.tsv', sep='\t')
assert len(questions) == 14267 and questions.id.is_unique
# PopQA phải đúng file của tác giả (bỏ khác biệt xuống dòng Windows/Linux trước khi so).
POPQA_SHA256 = '9a5227f41bff0e4c331d4a774d946b12f95307892b58f860a9606ef356e6089b'
assert hashlib.sha256(Path('data/popQA.tsv').read_bytes().replace(b'\r\n', b'\n')).hexdigest() == POPQA_SHA256, 'popQA.tsv khác bản của tác giả.'
print('Questions:', len(questions), 'Relations:', questions.prop.nunique())
print('Source SHA256:', hashlib.sha256(Path('run_model.py').read_bytes()).hexdigest())
print('Working directory:', work)

0 Tesla T4 14.6 GiB
1 Tesla T4 14.6 GiB
Đã tải mã nguồn và PopQA của tác giả từ GitHub (commit 8172ab0).
Questions: 14267 Relations: 16
Source SHA256: 5c87f0184792989612226c07c398f0953518f61e56cdf9fcd08f57ba5a6bc3a3
Working directory: /kaggle/working/popqa-qwen


In [3]:
# Danh sách cố định do nhóm cung cấp (giữ nguyên thứ tự).
WANT_IDS = [5139062, 4638370, 1353394, 5923312, 3356649, 4996780, 743705, 2357044, 4066277, 1642428, 4188191, 605897, 1473780, 1223424, 5999251, 5948044, 780737, 6378269, 994647, 4409246, 4522741, 837389, 2671483, 3804084, 1872067, 666610, 656905, 2250382, 1791448, 2485455, 6506411, 5179731, 5789578, 6365456, 5422818, 2174102, 5006891, 5781178, 2990323, 184846, 2556139, 5557556, 4330249, 2507186, 946336, 2691112, 2226731, 3974408, 2449796, 4921831, 1646720, 1342774, 1481181, 1977383, 3793259, 192261, 2987359, 237759, 264374, 3720231, 1237178, 2647530, 4234132, 2071152, 1306567, 1228958, 1594184, 3826516, 1194410, 5960486, 5560949, 4496891, 1387057, 232809, 4997435, 388189, 5580295, 5982132, 2105565, 1969958, 3106384, 246959, 2763878, 5303917, 3868066, 2281877, 4393083, 2899611, 1574139, 4613808, 1804783, 1187163, 3467441, 788195, 5839103, 3672517, 2068848, 2194088, 1860470, 1001346, 1225029, 776128, 2654404, 1391948, 3315030, 1116447, 3784290, 4331418, 1054401, 5187250, 2849306, 3755811, 832908, 1549444, 1616396, 1358512, 447830, 2254982, 5918313, 4666651, 596547, 1953459, 2647017, 1680080, 4684145, 3243026, 1316602, 5929687, 1001108, 935146, 2908490, 5920784, 2592338, 1486783, 6281394, 4270058, 2073781, 6004711, 5918653, 3956481, 2833871, 2435793, 5945781, 3423081, 4404592, 6547647, 5846736, 5937854, 3950149, 4763253, 3633815, 288745, 2059485, 4652935, 1932836, 2630371, 2213921, 977275, 2766419, 636616, 144024, 2085939, 1447840, 1758617, 3524273, 5915241, 37343, 3677957, 2963470, 3080610, 5773632, 2839162, 1660687, 6502612, 1332334, 5962937, 1571598, 3087676, 5669872, 4265255, 2229505, 5137613, 1913397, 5592430, 1074278, 3763572, 196395, 3276946, 5113116, 2374633, 1919697, 1107870, 5045808, 6374945, 2663309, 2311628, 5088311, 6042636, 614930, 432839, 3520627, 1402351, 843797, 75263, 35, 244122, 5777706, 474017, 6532951, 4532127, 3095296, 2886182, 1570230, 3675616, 4310070, 701352, 3450665, 186809, 3903448, 3952467, 5041070, 3514599, 6081120, 3543733, 3294045, 5547893, 4329831, 6533919, 5263647, 4801270, 6385442, 1162864, 4484472, 215156, 1504171, 5951099, 2223262, 5644258, 1855691, 1035763, 2756925, 1466602, 1868517, 6161854, 3422786, 4584501, 380223, 3981349, 2664757, 5485882, 3178175, 4405633, 1401285, 1349825, 3183040, 790430, 3845086, 1120961, 1134562, 285213, 632979, 353985, 285832, 1997293, 201733, 72369, 2897641, 595358, 4968395, 1001697, 1108230, 902091, 5970015, 144894, 398909, 2951170, 3172905, 3261768, 5169150, 500365, 5777300, 2114057, 1117503, 617409, 4365138, 394543, 691428, 1722398, 5770014, 6521985, 1081675, 2181196, 5979071, 4557797, 2961443, 1717233, 294568, 6001463, 2379460, 2245901]
EXPECTED_SOURCE_SHA256 = '5c87f0184792989612226c07c398f0953518f61e56cdf9fcd08f57ba5a6bc3a3'
assert len(WANT_IDS) == len(set(WANT_IDS)) == 300
assert set(WANT_IDS) <= set(questions.id)
assert hashlib.sha256(Path('run_model.py').read_text(encoding='utf-8').encode()).hexdigest() == EXPECTED_SOURCE_SHA256, 'Code nguồn khác bản đã kiểm tra; dừng để xem thay đổi.'
pd.Series(WANT_IDS).to_csv('subset_300_ids.txt', index=False, header=False)
import numpy as np
assert questions.sample(n=300, random_state=633).id.tolist() == WANT_IDS, 'PopQA/thứ tự dữ liệu không khớp subset cố định.'

## 2. Kiểm BM25 và tạo bản chạy có đoạn văn cho ví dụ mẫu

In [4]:
# Tìm file BM25. Nếu có nhiều file, điền đường dẫn chính xác ở đây.
import json
BM25_PATH = None

if BM25_PATH is None:
    candidates = sorted(
        p for p in Path('/kaggle/input').rglob('*.jsonl')
        if 'bm25' in str(p).lower() and '__MACOSX' not in str(p)
        and 'checkpoints' not in p.parts  # checkpoint của lượt full cũng là .jsonl có chữ BM25
    )
    if len(candidates) != 1:
        raise RuntimeError(
            'Hãy Add Input dataset chứa BM25 .jsonl đã giải nén, '
            'hoặc điền BM25_PATH chính xác. Các file tìm thấy:\n'
            + '\n'.join(map(str, candidates))
        )
    BM25_PATH = candidates[0]
else:
    BM25_PATH = Path(BM25_PATH)

assert BM25_PATH.is_file(), f'Không tìm thấy: {BM25_PATH}'
retrieval = {}
with BM25_PATH.open(encoding='utf-8') as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            retrieval[record['question']] = record

missing = [q for q in questions.question
           if not retrieval.get(q, {}).get('ctxs')
           or not retrieval[q]['ctxs'][0].get('text', '').strip()]
if missing:
    raise RuntimeError(f'Thiếu đoạn BM25 cho {len(missing)} câu. Ví dụ: {missing[:3]}')

# Kiểm mọi câu vì toàn bộ PopQA là nguồn lấy ví dụ few-shot.
for q in questions.question:
    ctx = retrieval[q]['ctxs'][0]
    assert {'text', 'id', 'hasanswer'} <= set(ctx), f'Thiếu trường BM25: {q}'
print(f'Đủ đoạn top-1 cho cả {len(questions):,} câu hỏi chính và nguồn ví dụ mẫu.')
del retrieval

# Chỉ tắt thanh tiến độ, giữ nguyên thuật toán và mã nguồn gốc.
source = (work / 'run_model.py').read_text(encoding='utf-8')
old = 'tqdm(sample.iloc, total=n)'
assert source.count(old) == 1, 'Code nguồn khác dự kiến; chưa tạo bản chạy im lặng.'
runner = work / 'run_model_bm25_shot_passages.py'
# Sửa duy nhất logic truy xuất few-shot: dùng nguyên văn câu hỏi như câu chính.
# Không tạo alias chuẩn hóa; không tự chọn passage khác.
import ast
tree = ast.parse(source)
fn = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'get_few_shot_text_with_retrieval')
lines = source.splitlines(keepends=True)
replacement = r"""def get_few_shot_text_with_retrieval(row, retrieval_dict, eval_method):
    if eval_method == "vanilla":
        return completion_template.format(row.question) + " " + row.obj
    retrieval = retrieval_dict[row.question]["ctxs"][0]
    if not retrieval["text"].strip():
        raise ValueError("Empty few-shot retrieval: " + row.question)
    return clip_paragraph(retrieval["text"], eval_method) + "\n\n" + completion_template.format(row.question) + " " + row.obj

"""
patched = ''.join(lines[:fn.lineno - 1]) + replacement + ''.join(lines[fn.end_lineno:])
quiet_source = patched.replace(old, 'tqdm(sample.iloc, total=n, disable=True)')
progress_anchor = '        accuracy.append(is_correct)'
assert quiet_source.count(progress_anchor) == 1, 'Không tìm thấy vị trí cập nhật tiến độ.'
progress_hook = """
        if len(accuracy) % 100 == 0:
            progress_path = os.environ.get('POPQA_PROGRESS_PATH')
            if progress_path:
                with open(progress_path + '.tmp', 'w') as progress_file:
                    json.dump({'done': len(accuracy)}, progress_file)
                os.replace(progress_path + '.tmp', progress_path)
"""
quiet_source = quiet_source.replace(progress_anchor, progress_anchor + progress_hook)
(work / 'checkpoint_utils.py').write_text("import json\nimport os\nfrom pathlib import Path\n\ndef load_checkpoint(path, run_key, ids):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    header = dict(version=1, run_key=run_key, ids=[int(i) for i in ids])\n    if not path.exists():\n        temp = path.with_suffix('.tmp')\n        with temp.open('w', encoding='utf-8') as f:\n            f.write(json.dumps(header) + '\\n')\n            f.flush(); os.fsync(f.fileno())\n        os.replace(temp, path)\n        return []\n    records = []\n    with path.open('rb+') as f:\n        first = f.readline()\n        if not first.endswith(b'\\n') or json.loads(first) != header:\n            raise ValueError('Checkpoint khác cấu hình hoặc danh sách ID; không tự ghi đè.')\n        while True:\n            offset = f.tell()\n            line = f.readline()\n            if not line:\n                break\n            if not line.endswith(b'\\n'):\n                # A killed write may leave only the final record incomplete.\n                f.seek(offset); f.truncate(); f.flush(); os.fsync(f.fileno())\n                break\n            record = json.loads(line)  # malformed complete records must fail loudly\n            i = len(records)\n            if i >= len(ids) or record['id'] != int(ids[i]):\n                raise ValueError('Checkpoint không phải prefix đúng thứ tự của tập chạy.')\n            if not all(k in record for k in ('prompt', 'pred', 'response', 'conf', 'is_correct')):\n                raise ValueError('Checkpoint thiếu trường kết quả.')\n            records.append(record)\n    return records\n\ndef append_checkpoint(path, record):\n    # Flush/fsync each completed answer; an interrupted last line is recoverable.\n    payload = (json.dumps(record, ensure_ascii=False, allow_nan=False) + '\\n').encode('utf-8')\n    with open(path, 'ab') as f:\n        f.write(payload); f.flush(); os.fsync(f.fileno())\n", encoding='utf-8')

# Checkpoint từng câu. Khi resume, dựng lại prompt các câu cũ để giữ đúng
# tiến trình random chọn few-shot; không gọi model.generate cho các câu đó.
checkpoint_import = 'from checkpoint_utils import load_checkpoint, append_checkpoint\n'
quiet_source = checkpoint_import + quiet_source
anchor = '    # main loop'
assert quiet_source.count(anchor) == 1
quiet_source = quiet_source.replace(anchor, "    checkpoint_path = os.environ.get('POPQA_CHECKPOINT_PATH') if args.sample == 0 else None\n    completed = load_checkpoint(checkpoint_path, os.environ['POPQA_RUN_KEY'], sample.id.tolist()) if checkpoint_path else []\n    if checkpoint_path:\n        print(f'Resume: {len(completed)}/{len(sample)} câu đã lưu; dựng lại prompt rồi tiếp tục.', flush=True)\n" + '\n' + anchor)
anchor = '        pred, response = generate(prompt, max_new_tokens=args.max_new_tokens)'
assert quiet_source.count(anchor) == 1
quiet_source = quiet_source.replace(anchor, '        if len(preds) < len(completed):\n            cached = completed[len(preds)]\n            if cached[\'prompt\'] != prompt:\n                raise ValueError(f"Prompt resume khác ở ID {row.id}; dừng để tránh trộn kết quả.")\n            pred, response = cached[\'pred\'], cached[\'response\']\n        else:\n            pred, response = generate(prompt, max_new_tokens=args.max_new_tokens)\n')
anchor = '        accuracy.append(is_correct)'
assert quiet_source.count(anchor) == 1
quiet_source = quiet_source.replace(anchor, anchor + '\n' + "        if checkpoint_path and len(preds) > len(completed):\n            append_checkpoint(checkpoint_path, dict(id=int(row.id), prompt=prompt,\n                pred=pred, response=response, is_correct=bool(is_correct)))\n        elif len(preds) <= len(completed) and bool(is_correct) != completed[len(preds) - 1]['is_correct']:\n            raise ValueError('Nhãn chấm điểm khác checkpoint.')\n")
# Disable upstream CSV snapshots: pilot has no checkpoints, full uses journal.
assert quiet_source.count('if len(preds) % 100 == 0:') == 1
quiet_source = quiet_source.replace('if len(preds) % 100 == 0:', 'if False:  # replaced by full-only JSONL checkpoint')

# Issue #14: ghi thêm cột `conf` = trung bình log-xác suất của các token tạo nên câu trả lời
# (dòng đầu của phần sinh ra, đúng phần code gốc lấy làm `pred`). Giải mã vẫn greedy, 15 token;
# chỉ bật output_scores nên prompt và dự đoán không đổi. Checkpoint lưu cả conf để chạy tiếp.
CONF_HELPER = '''
LAST_CONF = [None]  # conf của lần gọi call_model gần nhất


def answer_confidence(model, tokenizer, out):
    """Trung bình log-xác suất của các token thuộc câu trả lời; None nếu câu trả lời rỗng."""
    steps = len(out.scores)
    gen_ids = out.sequences[0, out.sequences.shape[1] - steps:]
    logp = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)[0]
    decode = lambda ids: tokenizer.decode(ids, clean_up_tokenization_spaces=False)
    text = decode(gen_ids)
    start = 2 if text.startswith("\\n\\n") else 0  # code gốc bỏ "\\n\\n" ở đầu
    end = text.find("\\n", start)                # rồi lấy tới hết dòng đầu
    end = len(text) if end < 0 else end
    picked, prev = [], 0
    for k in range(steps):
        cur = len(decode(gen_ids[:k + 1]))
        if cur > start and prev < end:
            picked.append(float(logp[k]))
        prev = cur
    return sum(picked) / len(picked) if picked else None


'''
conf_patches = [
    ('def call_model(', CONF_HELPER + 'def call_model('),
    ('num_beams=1, do_sample=False)\n    text = tokenizer.decode(gen[0])',
     'num_beams=1, do_sample=False, output_scores=True, return_dict_in_generate=True)\n'
     '    LAST_CONF[0] = answer_confidence(model, tokenizer, gen)\n'
     '    gen = gen.sequences\n    text = tokenizer.decode(gen[0])'),
    ('    preds = []\n', '    preds = []\n    confs = []\n'),
    ("            pred, response = cached['pred'], cached['response']\n",
     "            pred, response = cached['pred'], cached['response']\n            conf = cached['conf']\n"),
    ('            pred, response = generate(prompt, max_new_tokens=args.max_new_tokens)\n',
     '            pred, response = generate(prompt, max_new_tokens=args.max_new_tokens)\n'
     '            conf = LAST_CONF[0]\n'),
    ('        preds.append(pred)\n', '        preds.append(pred)\n        confs.append(conf)\n'),
    ('pred=pred, response=response, is_correct=bool(is_correct)))',
     'pred=pred, response=response, conf=conf, is_correct=bool(is_correct)))'),
    ('    sample["generation"] = responses\n',
     '    sample["generation"] = responses\n    sample["conf"] = confs\n'),
]
for old_text, new_text in conf_patches:
    assert quiet_source.count(old_text) == 1, f'Không tìm thấy chỗ vá conf: {old_text[:50]!r}'
    quiet_source = quiet_source.replace(old_text, new_text)

# Qwen đọc được ngữ cảnh rất dài, nên code gốc không cắt prompt cho Qwen. Nhóm chốt giữ cửa sổ
# 2048 token như GPT-Neo/OPT (cắt từ trái, dành 15 token cho câu trả lời), như code gốc làm với OPT.
window_old = 'if "opt" in args.model_name or args.model_name == "EleutherAI/gpt-neox-20b":'
assert quiet_source.count(window_old) == 1, 'Không tìm thấy chỗ đặt cửa sổ ngữ cảnh.'
quiet_source = quiet_source.replace(
    window_old, 'if "opt" in args.model_name or "qwen" in args.model_name.lower() or args.model_name == "EleutherAI/gpt-neox-20b":')

# Bản Kaggle mới (Python 3.13) rút 5 số ngẫu nhiên của numpy trong lúc import/nạp mô hình,
# sau dòng np.random.seed(633) của code gốc, nên tập câu lệch khỏi 300 ID cố định.
# Đặt lại seed 633 ngay trước khi chọn câu: tập câu và ví dụ mẫu giống code gốc
# chạy trên môi trường sạch (lượt đối chứng 26/10/2026 ra đúng 300 ID này).
sample_anchor = '    n = len(knowledge) if args.sample == 0 else args.sample'
assert quiet_source.count(sample_anchor) == 1, 'Không tìm thấy chỗ chọn câu trong code gốc.'
quiet_source = quiet_source.replace(
    sample_anchor, '    random.seed(seed)\n    np.random.seed(seed)\n' + sample_anchor)

runner.write_text(quiet_source, encoding='utf-8')
ast.parse(quiet_source)
print('Đã tạo runner: few-shot BM25 có đoạn văn, có cột conf, Qwen cắt ở 2048 token, ghi tiến độ mỗi 100 câu.')

Đủ đoạn top-1 cho cả 14,267 câu hỏi chính và nguồn ví dụ mẫu.
Đã tạo runner: few-shot BM25 có đoạn văn, có cột conf, Qwen cắt ở 2048 token, ghi tiến độ mỗi 100 câu.


## 3. Hàm chạy và kiểm tra kết quả
Mỗi lượt một log riêng, có trạng thái cập nhật mỗi phút. Chỉ lượt full lưu checkpoint từng câu và tự chạy tiếp. Lượt 300 câu chỉ lưu CSV cuối.

In [5]:
import time
from IPython.display import display

import platform, transformers, jsonlines, importlib.metadata

def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

# Mỗi lượt = một mô hình × một cách chạy. Mỗi GPU một hàng đợi; hai GPU chạy song song.
# Qwen 3B BM25 lâu nhất (prompt dài vì có đoạn văn) nên giữ riêng GPU 0; GPU 1 chạy vanilla.
JOBS = [
    dict(model='Qwen/Qwen2.5-3B', method='BM25', gpu=0),
    dict(model='Qwen/Qwen2.5-3B', method='vanilla', gpu=1),
]
for job in JOBS:
    job['name'] = f"{job['model']} · {job['method']}"
JOB_NAMES = [job['name'] for job in JOBS]

# Cấu hình ảnh hưởng tới kết quả. Đổi bất kỳ thứ gì ở đây thì RUN_KEY đổi và checkpoint cũ bị từ chối.
RUN_CONFIG = dict(shot_passages=True, conf=True, context_window=2048, seed=633, n_shots=15, max_new_tokens=15,
    dtype='fp32 (upstream default)', decoding='greedy',
    source_sha256=file_sha(work / 'run_model.py'), runner_sha256=file_sha(runner), checkpoint_helper_sha256=file_sha(work / 'checkpoint_utils.py'),
    popqa_sha256=file_sha(work / 'data/popQA.tsv'), bm25_sha256=file_sha(BM25_PATH),
    subset_ids=WANT_IDS, python=platform.python_version(),
    packages={p: importlib.metadata.version(p) for p in ['torch', 'transformers', 'numpy', 'pandas', 'jsonlines']},
    gpu=torch.cuda.get_device_name(0))
RUN_KEY = hashlib.sha256(json.dumps(RUN_CONFIG, sort_keys=True).encode()).hexdigest()
(work / 'run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')


def read_result(path, expected_n, method):
    frame = pd.read_csv(path, keep_default_na=False)
    assert len(frame) == expected_n and frame.id.is_unique, f'CSV chưa đầy đủ: {path}'
    assert set(frame.id) <= set(questions.id), 'Có ID ngoài dữ liệu PopQA.'
    need = {'pred', 'is_correct', 'conf'} | ({'retrieval_id', 'has_answer'} if method == 'BM25' else set())
    assert need <= set(frame.columns), f'Thiếu cột {sorted(need - set(frame.columns))}: {path}'
    flags = frame.is_correct.astype(str).str.strip().str.lower()
    assert flags.isin(['true', 'false']).all(), 'Nhãn đúng/sai không hợp lệ.'
    frame['is_correct'] = flags.eq('true')
    return frame

In [6]:
# Mỗi lượt chạy một tiến trình riêng (seed đặt lại như code gốc), nên thứ tự câu và ví dụ mẫu
# giống hệt khi chạy một mình. Lượt đầu của GPU 0 lên GPU xong mới nạp lượt đầu của GPU 1, để hai
# lần nạp fp32 không cùng chiếm RAM. Chỉ có 1 GPU thì mọi lượt chạy lần lượt trên GPU đó.
N_GPUS = torch.cuda.device_count()
for job in JOBS:
    job['device'] = f"cuda:{job['gpu'] % N_GPUS}"
# Kaggle cắt phiên ở 12 giờ. Dừng gọn trước đó để còn giờ kiểm và đóng gói checkpoint.
TIME_BUDGET_HOURS = 8.0
POLL_SECONDS = 60
print('Phân việc:', *[f"  {job['device']}: {job['name']}" for job in JOBS], sep='\n')


def job_alias(job, suffix):
    """Tên lượt trong tên file. BM25 giữ tên như lượt GPT-Neo trước (ví dụ mẫu kèm đoạn văn)."""
    return f'popqa-shot-passages-{suffix}' if job['method'] == 'BM25' else f'popqa-{suffix}'


def start_run(job, sample, suffix):
    """Khởi động code gốc cho một lượt. Đã có CSV đầy đủ thì trả về run với process=None."""
    assert job in JOBS
    assert sample in [0, 300]
    model_name, method, device = job['model'], job['method'], job['device']
    alias = job_alias(job, suffix)
    expected_n = len(questions) if sample == 0 else sample
    filename = (
        f'model={model_name.replace("/", "_")}-input={alias}'
        f'-method={method}-shots=15-n={expected_n}.csv'
    )
    destination = work / 'results' / filename
    meta_path = destination.with_suffix('.run.json')
    run = dict(job=job, name=job['name'], model=model_name, method=method, device=device, sample=sample,
               alias=alias, expected_n=expected_n, destination=destination, meta_path=meta_path, process=None)
    if destination.exists():
        assert meta_path.exists() and json.loads(meta_path.read_text())['run_key'] == RUN_KEY, 'CSV cũ khác cấu hình/không có metadata; dùng thư mục làm việc mới.'
        read_result(destination, expected_n, method)
        print('Đã có CSV đầy đủ, bỏ qua:', destination.name)
        return run

    command = [sys.executable, '-u', str(runner),
               '--model_name', model_name,
               '--input_file', str(work / 'data/popQA.tsv'), '--alias', alias,
               '--eval_method', method, '--ret_path', str(BM25_PATH),
               '--n_examples', '15', '--sample', str(sample),
               '--device', device, '--max_new_tokens', '15']
    log_path = work / 'logs' / (filename + '.log')
    progress_path = work / 'logs' / (filename + '.progress.json')
    if not progress_path.exists():
        progress_path.write_text('{"done": 0}', encoding='utf-8')
    env = os.environ.copy()
    env['POPQA_PROGRESS_PATH'] = str(progress_path)
    env.pop('POPQA_CHECKPOINT_PATH', None)
    if sample == 0:
        env['POPQA_CHECKPOINT_PATH'] = str(work / 'checkpoints' / (filename + '.jsonl'))
    env['POPQA_RUN_KEY'] = RUN_KEY
    # Metadata written before launch also permits recovery after final CSV is saved.
    meta_path.write_text(json.dumps(dict(run_key=RUN_KEY, model=model_name, method=method, sample=sample, alias=alias)), encoding='utf-8')
    print(f'Bắt đầu {job["name"]} trên {device} — {expected_n:,} câu. Log: {log_path}', flush=True)
    log = log_path.open('a', encoding='utf-8')
    run.update(log=log, log_path=log_path, progress_path=progress_path,
               progress_mtime=progress_path.stat().st_mtime_ns, started=time.monotonic(),
               status=display(f'{job["name"]} | {device}: đang nạp mô hình...', display_id=True))
    run['process'] = subprocess.Popen(command, cwd=work, stdout=log, stderr=subprocess.STDOUT, env=env)
    return run


def show_progress(run, done=None, note=''):
    if done is None:
        done = json.loads(run['progress_path'].read_text(encoding='utf-8'))['done']
    elapsed = int(time.monotonic() - run['started'])
    hours, rem = divmod(elapsed, 3600)
    minutes, seconds = divmod(rem, 60)
    run['status'].update(
        f'{run["name"]} | {run["device"]} | {done:,}/{run["expected_n"]:,} câu '
        f'({done / run["expected_n"]:.1%}) | Đã chạy {hours:02}:{minutes:02}:{seconds:02}{note}'
    )


def wait_until_loaded(run, limit=1800):
    """Đợi lần ghi tiến độ đầu tiên (mô hình đã lên GPU) rồi mới nạp mô hình kế."""
    started = time.monotonic()
    while (run['process'].poll() is None and time.monotonic() - started < limit
           and run['progress_path'].stat().st_mtime_ns == run['progress_mtime']):
        time.sleep(POLL_SECONDS / 4)


def stop_process(process):
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()


def checkpoint_count(run):
    """Số câu đã ghi trọn vào checkpoint (trừ dòng đầu chứa cấu hình)."""
    path = work / 'checkpoints' / (run['destination'].name + '.jsonl')
    if not path.exists():
        return 0
    with path.open('rb') as f:
        return max(sum(1 for line in f if line.endswith(b'\n')) - 1, 0)


def finish_run(run, code):
    """Đóng một lượt đã thoát. Trả về câu báo lỗi, hoặc None nếu CSV đầy đủ."""
    run['log'].close()
    if code != 0:
        from collections import deque
        with run['log_path'].open(encoding='utf-8', errors='replace') as log:
            print(''.join(deque(log, maxlen=25)))
        return f'{run["name"]}: chạy lỗi (exit {code}). Xem {run["log_path"]}'
    saved = read_result(run['destination'], run['expected_n'], run['method'])
    run['meta_path'].write_text(json.dumps(dict(run_key=RUN_KEY, model=run['model'], method=run['method'], sample=run['sample'], alias=run['alias'])), encoding='utf-8')
    show_progress(run, len(saved))
    print(f'Hoàn thành {run["name"]}: accuracy = {saved.is_correct.mean():.2%}')
    return None


def run_models(sample, suffix, deadline=None):
    """Chạy mọi lượt trong JOBS, mỗi GPU một hàng đợi.

    Trả về (CSV đã xong theo tên lượt, danh sách lượt tạm dừng vì hết giờ).
    Một lượt lỗi không làm dừng các lượt khác; báo lỗi sau khi mọi lượt khác xong.
    """
    queues = {}
    for job in JOBS:
        queues.setdefault(job['device'], []).append(job)
    done, paused, errors, active = {}, [], [], []

    def launch(device):
        # Khởi động lượt kế tiếp của GPU này; bỏ qua lượt đã có CSV đầy đủ.
        while queues[device]:
            run = start_run(queues[device].pop(0), sample, suffix)
            if run['process'] is None:
                done[run['name']] = run['destination']
            else:
                active.append(run)
                return run
        return None

    try:
        for device in queues:
            run = launch(device)
            if run is not None:
                wait_until_loaded(run)
        while active:
            if deadline is not None and time.time() >= deadline:
                for run in active:
                    stop_process(run['process'])
                    run['log'].close()
                    saved = checkpoint_count(run)
                    show_progress(run, saved, ' | TẠM DỪNG: sắp hết giờ phiên')
                    print(f'Tạm dừng {run["name"]}: đã lưu {saved:,}/{run["expected_n"]:,} câu vào checkpoint.')
                    paused.append(run['name'])
                for device in queues:  # lượt chưa kịp bắt đầu cũng để phiên sau
                    paused.extend(job['name'] for job in queues[device])
                active.clear()
                break
            time.sleep(POLL_SECONDS)
            for run in list(active):
                code = run['process'].poll()
                if code is None:
                    show_progress(run)
                    continue
                active.remove(run)
                error = finish_run(run, code)
                if error:
                    errors.append(error)
                else:
                    done[run['name']] = run['destination']
                launch(run['device'])
    except BaseException:
        for run in active:
            stop_process(run['process'])
            run['log'].close()
        raise
    if errors:
        raise RuntimeError('\n'.join(errors))
    return done, paused

Phân việc:
  cuda:0: Qwen/Qwen2.5-3B · BM25
  cuda:1: Qwen/Qwen2.5-3B · vanilla


### Checkpoint và chạy tiếp — chỉ dành cho full dataset
Lượt 300 câu không lưu checkpoint; bị ngắt thì chạy lại lượt thử. Ở lượt full, mỗi câu xong được ghi ngay vào `checkpoints/*.jsonl` và đồng bộ xuống đĩa. Chạy lại thì mô hình chỉ sinh các câu còn thiếu. Các câu cũ được dựng lại prompt để giữ đúng thứ tự bốc ví dụ ngẫu nhiên; prompt dựng lại phải trùng từng chữ với bản đã lưu.

Lượt full tự dừng gọn sau `TIME_BUDGET_HOURS` giờ và đóng gói `popqa-qwen-checkpoint.zip`. Sang phiên mới: Add Input file ZIP đó (Output của phiên trước, hoặc dataset chứa nó) rồi chạy lại toàn bộ notebook. `RESTORE_FROM = 'auto'` tự tìm và khôi phục. Cần cùng dữ liệu, mã, bản thư viện và GPU; khác thì notebook dừng, không tự trộn kết quả.

In [7]:
# 'auto': tự tìm checkpoint của phiên trước trong Input: file popqa-qwen-checkpoint.zip,
# hoặc thư mục gốc có results/*.run.json. Có thể điền đường dẫn cụ thể; None = không khôi phục.
RESTORE_FROM = 'auto'
if RESTORE_FROM == 'auto':
    input_root = Path('/kaggle/input')
    found = sorted(input_root.rglob('popqa-qwen-checkpoint.zip'))
    if not found:
        found = sorted({p.parent.parent for p in input_root.rglob('*.run.json') if p.parent.name == 'results'})
    if len(found) > 1:
        raise RuntimeError('Có nhiều bản checkpoint trong Input; điền RESTORE_FROM chính xác:\n'
                           + '\n'.join(map(str, found)))
    RESTORE_FROM = found[0] if found else None
    print('Khôi phục từ:', RESTORE_FROM if RESTORE_FROM else 'không có, chạy mới.')
if RESTORE_FROM is not None:
    import zipfile
    origin = Path(RESTORE_FROM)
    def restore_one(rel, data):
        rel = Path(rel)
        if rel.is_absolute() or '..' in rel.parts or not rel.parts:
            raise ValueError('Đường dẫn checkpoint không hợp lệ.')
        if rel.parts[0] not in {'checkpoints', 'results', 'logs'}:
            return
        target = work / rel
        if target.exists():
            assert target.read_bytes() == data, f'File đích khác bản khôi phục: {rel}. Dùng thư mục làm việc mới.'
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
    if origin.is_file():
        with zipfile.ZipFile(origin) as z:
            for info in z.infolist():
                if not info.is_dir():
                    restore_one(info.filename, z.read(info))
    elif origin.is_dir():
        # Chọn thư mục gốc có checkpoints/, results/, logs/.
        for sub in ('checkpoints', 'results', 'logs'):
            for p in (origin / sub).rglob('*'):
                if p.is_file():
                    restore_one(p.relative_to(origin), p.read_bytes())
    else:
        raise FileNotFoundError(origin)
    print('Đã khôi phục. Cấu hình và thứ tự ID sẽ được kiểm khi chạy.')


Khôi phục từ: /kaggle/input/datasets/chanejlaforet/popqa-qwen3b-checkpoint
Đã khôi phục. Cấu hình và thứ tự ID sẽ được kiểm khi chạy.


## 4. Chạy thử 300 câu cố định cho cả bốn lượt Qwen
Kiểm thử kỹ thuật trước khi chạy đủ: đúng 300 ID, đúng thứ tự, prompt dựng lại khớp từng chữ. Có checkpoint khôi phục thì bỏ qua lượt đã có CSV đầy đủ.

In [8]:
PILOT_SUFFIX = 'check300'
pilot_paths, _ = run_models(sample=300, suffix=PILOT_SUFFIX)
assert set(pilot_paths) == set(JOB_NAMES), 'Lượt 300 câu chưa đủ mọi lượt.'

Đã có CSV đầy đủ, bỏ qua: model=Qwen_Qwen2.5-3B-input=popqa-shot-passages-check300-method=BM25-shots=15-n=300.csv
Đã có CSV đầy đủ, bỏ qua: model=Qwen_Qwen2.5-3B-input=popqa-check300-method=vanilla-shots=15-n=300.csv


## 5. Đánh giá 300 câu và kiểm lỗi
Kiểm đủ ID và thứ tự; không thiếu đoạn BM25; dựng lại toàn bộ prompt (15 ví dụ mẫu) và so từng chữ. Báo accuracy, câu trả lời rỗng, số prompt vượt giới hạn và số dòng có `conf`.

In [9]:
from transformers import AutoTokenizer
import random

# Đoạn top-1 BM25 theo nguyên văn câu hỏi, chỉ đọc khi kiểm lượt BM25.
_BM25_CACHE = {}

def bm25_lookup():
    if not _BM25_CACHE:
        with BM25_PATH.open(encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    d = json.loads(line); _BM25_CACHE[d['question']] = d
    return _BM25_CACHE


def verify_pilot(job, path):
    """Kiểm một lượt 300 câu: đúng 300 ID theo thứ tự, và dựng lại được từng prompt từng chữ."""
    method = job['method']
    res = read_result(path, 300, method)
    assert res.id.astype(int).tolist() == WANT_IDS, f'{job["name"]}: không đúng 300 ID/thứ tự; không chạy full.'
    assert json.loads(path.with_suffix('.run.json').read_text())['run_key'] == RUN_KEY
    log = (work / 'logs' / (path.name + '.log')).read_text(encoding='utf-8', errors='replace')
    assert 'missing retrieval' not in log and 'No retrieval for' not in log, 'Có lookup BM25 thất bại.'

    # Chỉ nạp các hàm tạo prompt và template từ runner, không nạp model trên CPU.
    tree = ast.parse(runner.read_text(encoding='utf-8'))
    wanted = {'clip_paragraph', 'get_few_shot_text_with_retrieval', 'get_few_shot_text'}
    nodes = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name in wanted]
    nodes += [n for n in tree.body if isinstance(n, ast.Assign)
              and any(isinstance(t, ast.Name) and t.id in {'q_templates', 'completion_template'} for t in n.targets)]
    ns = {}
    exec(compile(ast.Module(body=nodes, type_ignores=[]), '<prompt-check>', 'exec'), ns)
    ret = bm25_lookup() if method == 'BM25' else None
    np.random.seed(633)
    random.seed(633)
    sampled = questions.sample(n=300, replace=False)
    for row, actual in zip(sampled.itertuples(index=False), res.itertuples(index=False)):
        shots = []
        for pid in ns['q_templates']:
            if pid == row.prop_id:
                continue
            for demo in questions[questions.prop_id == pid].sample(n=1).itertuples(index=False):
                if method == 'BM25':
                    shot = ns['get_few_shot_text_with_retrieval'](demo, ret, 'BM25')
                    assert shot.startswith(ret[demo.question]['ctxs'][0]['text'] + '\n\n')
                else:
                    shot = ns['get_few_shot_text'](demo, method)
                shots.append(shot)
        assert len(shots) == 15
        np.random.shuffle(shots)
        head = '\n\n'.join(shots) + '\n\n'
        if method == 'BM25':
            ctx = ret[row.question]['ctxs'][0]
            expected = head + ctx['text'] + '\n\n' + ns['completion_template'].format(row.question)
            assert str(actual.retrieval_id) == str(ctx['id']), f'Sai đoạn chính ở ID {row.id}'
        else:
            expected = head + ns['completion_template'].format(row.question)
        assert actual.prompt == expected, f'{job["name"]}: prompt không khớp ở ID {row.id}; dừng để kiểm tra.'
    tokenizer = AutoTokenizer.from_pretrained(job['model'])
    lengths = [len(tokenizer(p, truncation=False)['input_ids']) for p in res.prompt]
    budget = min(tokenizer.model_max_length, 2048) - 15
    conf = pd.to_numeric(res.conf.replace('', np.nan))
    return dict(model=job['model'], method=method, n=300, accuracy=float(res.is_correct.mean()),
                mean_prompt_tokens=float(np.mean(lengths)), max_prompt_tokens=max(lengths),
                truncated_prompts=sum(n > budget for n in lengths),
                empty_predictions=int(res.pred.astype(str).str.strip().eq('').sum()),
                conf_filled=int(conf.notna().sum()), mean_conf=float(conf.mean()),
                ids_and_prompts_ok=True)

pilot_summary = pd.DataFrame([verify_pilot(job, pilot_paths[job['name']]) for job in JOBS])
pilot_summary.to_csv(work / 'results/pilot300-summary.csv', index=False)
display(pilot_summary)
print('PASS: mọi lượt đúng 300 ID, đúng thứ tự, prompt dựng lại khớp từng chữ.')
vanilla_job = next(job for job in JOBS if job['method'] == 'vanilla')
print('Ví dụ prompt vanilla:')
print(read_result(pilot_paths[vanilla_job['name']], 300, 'vanilla').iloc[0]['prompt'])

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

,model,method,n,accuracy,mean_prompt_tokens,max_prompt_tokens,truncated_prompts,empty_predictions,conf_filled,mean_conf,ids_and_prompts_ok
0,Qwen/Qwen2.5-3B,BM25,300,0.293333,2102.98,3054,161,0,300,-0.745773,True
1,Qwen/Qwen2.5-3B,vanilla,300,0.193333,262.33,300,0,0,300,-1.527447,True


PASS: mọi lượt đúng 300 ID, đúng thứ tự, prompt dựng lại khớp từng chữ.
Ví dụ prompt vanilla:
Q: Who is the author of The Reader? A: Bernhard Schlink

Q: What sport does Olympique Lyonnais Féminin play? A: association football

Q: What is Pietro Marcenaro's occupation? A: politician

Q: Who was the composer of Serial? A: Lalo Schifrin

Q: Who was the producer of The Rainbow? A: Ken Russell

Q: What genre is The Interview? A: comedy film

Q: Who was the director of Connected? A: Benny Chan

Q: In what country is Valdearcos de la Vega? A: Spain

Q: Who is the father of Rahy? A: Blushing Groom

Q: In what city was Gianni Lonzi born? A: Florence

Q: What color is Stoke City F.C.? A: red

Q: What is the religion of Stephen? A: Christianity

Q: Who was the screenwriter for Kitty? A: Karl Tunberg

Q: What is the capital of Plaza? A: Guarenas

Q: What is Rome the capital of? A: Italy

Q: Who is the mother of Melissa Benn? A:


## 6. Chạy đủ 14.267 câu
Cell này chỉ chạy khi lượt 300 câu đã PASS. Phiên Kaggle tối đa 12 giờ, nên notebook tự dừng gọn sau `TIME_BUDGET_HOURS` giờ, đóng gói checkpoint và in lượt nào chưa xong. Phiên sau: Add Input `popqa-qwen-checkpoint.zip` của phiên này rồi chạy lại toàn bộ notebook.
Không ghép CSV 300 câu vào bản đủ: code gốc tiêu thụ số ngẫu nhiên khác khi lấy mẫu 300 và toàn bộ.

In [10]:
RUN_FULL = True
FULL_SUFFIX = 'full'
if RUN_FULL:
    assert pilot_summary.ids_and_prompts_ok.all() and len(pilot_summary) == len(JOBS), 'Lượt 300 câu chưa PASS.'
    deadline = SESSION_START + TIME_BUDGET_HOURS * 3600
    print(f'Phiên này tự dừng sau {TIME_BUDGET_HOURS} giờ kể từ lúc mở; còn lại {(deadline - time.time()) / 3600:.1f} giờ.')
    full_done, full_paused = run_models(sample=0, suffix=FULL_SUFFIX, deadline=deadline)
    if full_paused:
        print('Chưa xong:', ', '.join(full_paused) + '.',
              'Phiên sau: Add Input popqa-qwen-checkpoint.zip của phiên này rồi chạy lại toàn bộ notebook.')
    else:
        print('Đã xong đủ 14.267 câu cho mọi lượt.')
else:
    print('Chưa chạy full. Sau khi lượt 300 câu PASS, đổi RUN_FULL = True.')

Phiên này tự dừng sau 8.0 giờ kể từ lúc mở; còn lại 8.0 giờ.
Bắt đầu Qwen/Qwen2.5-3B · BM25 trên cuda:0 — 14,267 câu. Log: /kaggle/working/popqa-qwen/logs/model=Qwen_Qwen2.5-3B-input=popqa-shot-passages-full-method=BM25-shots=15-n=14267.csv.log


'Qwen/Qwen2.5-3B · BM25 | cuda:0 | 12,978/14,267 câu (91.0%) | Đã chạy 07:59:32 | TẠM DỪNG: sắp hết giờ phiên'

Đã có CSV đầy đủ, bỏ qua: model=Qwen_Qwen2.5-3B-input=popqa-full-method=vanilla-shots=15-n=14267.csv
Tạm dừng Qwen/Qwen2.5-3B · BM25: đã lưu 12,978/14,267 câu vào checkpoint.
Chưa xong: Qwen/Qwen2.5-3B · BM25. Phiên sau: Add Input popqa-qwen-checkpoint.zip của phiên này rồi chạy lại toàn bộ notebook.


## 7. Kiểm tra kết quả full và đóng gói
ZIP lưu riêng CSV, log, metadata, danh sách ID và code đã dùng. Có thể chạy cell đóng gói ngay sau pilot để tải kết quả thử về.

In [11]:
full_rows = []
for job in JOBS:
    alias = job_alias(job, 'full')
    path = work / 'results' / f'model={job["model"].replace("/", "_")}-input={alias}-method={job["method"]}-shots=15-n=14267.csv'
    if not path.exists():
        continue
    res = read_result(path, 14267, job['method'])
    assert set(res.id) == set(questions.id)
    assert json.loads(path.with_suffix('.run.json').read_text())['run_key'] == RUN_KEY
    log = (work / 'logs' / (path.name + '.log')).read_text(encoding='utf-8', errors='replace')
    assert 'missing retrieval' not in log and 'No retrieval for' not in log
    conf = pd.to_numeric(res.conf.replace('', np.nan))
    full_rows.append(dict(model=job['model'], method=job['method'], n=len(res),
                          accuracy=res.is_correct.mean(), conf_filled=int(conf.notna().sum())))
if full_rows:
    full_summary = pd.DataFrame(full_rows)
    full_summary.to_csv(work / 'results/full-summary.csv', index=False)
    display(full_summary)
else:
    print('Chưa có kết quả full; vẫn có thể tải ZIP lượt 300 câu ở cell sau.')

,model,method,n,accuracy,conf_filled
0,Qwen/Qwen2.5-3B,vanilla,14267,0.1835,14267


In [12]:
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink
zip_path = Path('/kaggle/working/popqa-qwen-results.zip')
files = sorted((work / 'results').glob('*.csv')) + sorted((work / 'results').glob('*.run.json'))
assert files, 'Chưa có kết quả để đóng gói.'
files += sorted((work / 'logs').glob('*.log'))
files += sorted((work / 'checkpoints').glob('*.jsonl'))
files += [work / 'checkpoint_utils.py']
files += [work / 'run_model.py', runner, work / 'util_clm.py', work / 'data/popQA.tsv',
          work / 'run_config.json', work / 'subset_300_ids.txt']
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, arcname=str(p.relative_to(work)))
with ZipFile(zip_path) as z:
    assert z.testzip() is None
os.chdir('/kaggle/working')
print('Đã lưu:', zip_path, '— tải ZIP để giữ kết quả.')
display(FileLink(zip_path.name))

Đã lưu: /kaggle/working/popqa-qwen-results.zip — tải ZIP để giữ kết quả.


/kaggle/working/popqa-qwen-results.zip

## Tải checkpoint để chuyển session
Lượt full tự chạy cell này khi tạm dừng vì hết giờ. Có thể chạy riêng cell này sau khi dừng lượt full, không cần chạy các cell đánh giá/đóng gói kết quả phía trên.

In [13]:
# Có thể chạy cell này sau khi Stop/Interrupt cell chạy mô hình.
# Không cần chờ chạy full xong mới tải được checkpoint.
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import FileLink, display
checkpoint_zip = Path('/kaggle/working/popqa-qwen-checkpoint.zip')
items = [p for sub in ('checkpoints', 'results', 'logs') for p in (work / sub).rglob('*') if p.is_file()]
assert items, 'Chưa có checkpoint/kết quả để tải.'
items += [p for p in [work / 'run_config.json', work / 'subset_300_ids.txt'] if p.exists()]
with ZipFile(checkpoint_zip, 'w', ZIP_DEFLATED) as z:
    for p in items:
        z.write(p, str(p.relative_to(work)))
os.chdir('/kaggle/working')
display(FileLink(checkpoint_zip.name))
print('Giữ ZIP này. Phiên mới: Add Input ZIP này rồi chạy lại toàn bộ notebook; RESTORE_FROM = \'auto\' tự khôi phục.')


/kaggle/working/popqa-qwen-checkpoint.zip

Giữ ZIP này. Phiên mới: Add Input ZIP này rồi chạy lại toàn bộ notebook; RESTORE_FROM = 'auto' tự khôi phục.
